In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.datasets import load_breast_cancer

In [ ]:
data = load_breast_cancer()
data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]]),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0

In [ ]:
x = data.data
y = data.target

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,random_state=42,test_size=0.2)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
xtrain = scaler.fit_transform(xtrain)
xtest = scaler.fit_transform(xtest)

In [ ]:
print("\n====================================================")
print("STEP 2 : TUNING OPTIMIZER")
print("====================================================")


STEP 2 : TUNING OPTIMIZER


In [ ]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.4 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

In [ ]:
xtrain.shape

(455, 30)

In [ ]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(64,activation="relu",input_dim=xtrain.shape[1]))
  model.add(Dense(32,activation="relu"))
  model.add(Dense(1,activation="sigmoid"))

  optimizer_name = hp.Choice(
      "Optimizers", values=["adam","rmsprop","adagrad","sgd"]
  )
  if optimizer_name=="adam":
    optimzer = Adam()
  elif optimizer_name=="rmsprop":
    optimzer = RMSprop()
  elif optimizer_name=="adagrad":
    optimer = Adagrad()
  else:
    optimizer = SGD()

  model.compile(loss="binary_crossentropy",optimizer=optimzer,metrics=["accuracy"])

  return model



In [ ]:
optimer_tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=4,
    directory="Keras Tuner",
    project_name="OptimizerTuner"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
optimer_tuner.search(
    xtrain,
    ytrain,
    epochs=10,
    validation_split=0.2
)

Trial 4 Complete [00h 00m 01s]

Best val_accuracy So Far: 0.9890109896659851
Total elapsed time: 00h 00m 10s


In [ ]:
best_optimizer = optimer_tuner.get_best_hyperparameters(1)[0].get("Optimizers")

In [ ]:
best_optimizer

'rmsprop'

In [ ]:
best_model = optimer_tuner.get_best_models()[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
best_model.fit(xtrain,ytrain,epochs=20,validation_split=0.2,initial_epoch=10)

Epoch 11/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.9835 - loss: 0.0976 - val_accuracy: 0.9780 - val_loss: 0.1144
Epoch 12/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9863 - loss: 0.0739 - val_accuracy: 0.9780 - val_loss: 0.1035
Epoch 13/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9890 - loss: 0.0634 - val_accuracy: 0.9780 - val_loss: 0.0976
Epoch 14/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9863 - loss: 0.0560 - val_accuracy: 0.9780 - val_loss: 0.0943
Epoch 15/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9890 - loss: 0.0499 - val_accuracy: 0.9780 - val_loss: 0.0902
Epoch 16/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9890 - loss: 0.0458 - val_accuracy: 0.9780 - val_loss: 0.0885
Epoch 17/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9890 - loss: 0.0420 - val_accuracy: 0.9670 - val_loss: 0.0870
Epoch 18/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9918 - loss: 0.0379 - val_accuracy: 0.9670 - 

In [ ]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,196 (32.02 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,099 (16.02 KB)

In [ ]:

print("\n====================================================")
print("STEP 3 : TUNING NUMBER OF NEURONS")
print("====================================================")



STEP 3 : TUNING NUMBER OF NEURONS


In [ ]:

def build_neuron_model(hp):
    model = Sequential()

    neurons_1 = hp.Int(
        'neurons_layer1',
        min_value=16,
        max_value=256,
        step=16
    )

    neurons_2 = hp.Int(
        'neurons_layer2',
        min_value=16,
        max_value=256,
        step=16
    )

    model.add(Dense(neurons_1, activation='relu', input_shape=(xtrain.shape[1],)))
    model.add(Dense(neurons_2, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=best_optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
neuron_tuner = kt.RandomSearch(
    build_neuron_model,
    objective='val_accuracy',
    max_trials=10,
    directory='Keras_Tuner',
    project_name='neuron_tuning'
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
neuron_tuner.search(
    xtrain,
    ytrain,
    validation_split=0.2,
    epochs=10,
)

Trial 10 Complete [00h 00m 03s]
val_accuracy: 0.9780219793319702

Best val_accuracy So Far: 0.9780219793319702
Total elapsed time: 00h 00m 33s


In [ ]:
best_neurons_hp = neuron_tuner.get_best_hyperparameters(1)[0]

best_neurons_layer1 = best_neurons_hp.get('neurons_layer1')
best_neurons_layer2 = best_neurons_hp.get('neurons_layer2')

print("\nBest Neurons for Layer 1:", best_neurons_layer1)
print("Best Neurons for Layer 2:", best_neurons_layer2)


Best Neurons for Layer 1: 224
Best Neurons for Layer 2: 240


In [ ]:
print("\n====================================================")
print("STEP 4 : TUNING NUMBER OF HIDDEN LAYERS")
print("====================================================")



STEP 4 : TUNING NUMBER OF HIDDEN LAYERS


In [ ]:
def build_layer_model(hp):
    model = Sequential()

    num_layers = hp.Int(
        'num_hidden_layers',
        min_value=1,
        max_value=5,
        step=1
    )

    model.add(Dense(best_neurons_layer1,
                    activation='relu',
                    input_shape=(xtrain.shape[1],)))

    for i in range(num_layers):
        model.add(Dense(best_neurons_layer2, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=best_optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
layer_tuner = kt.RandomSearch(
    build_layer_model,
    objective='val_accuracy',
    max_trials=5,
    directory='Keras Tuner',
    project_name='layer_tuning'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
layer_tuner.search(
    xtrain,
    ytrain,
    validation_split=0.2,
    epochs=10,
    verbose=1
)

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.9780219793319702

Best val_accuracy So Far: 0.9890109896659851
Total elapsed time: 00h 00m 20s


In [ ]:
best_layers = layer_tuner.get_best_hyperparameters(1)[0].get('num_hidden_layers')

print("\nBest Number of Hidden Layers:", best_layers)



Best Number of Hidden Layers: 3


In [ ]:
print("\n====================================================")
print("STEP 5 : TUNING ACTIVATION FUNCTION")
print("====================================================")


STEP 5 : TUNING ACTIVATION FUNCTION


In [ ]:
def build_activation_model(hp):
    model = Sequential()

    activation_function = hp.Choice(
        'activation_function',
        values=['relu', 'tanh', 'sigmoid']
    )

    model.add(Dense(best_neurons_layer1,
                    activation=activation_function,
                    input_shape=(xtrain.shape[1],)))

    for _ in range(best_layers):
        model.add(Dense(best_neurons_layer2,
                        activation=activation_function))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=best_optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:

activation_tuner = kt.RandomSearch(
    build_activation_model,
    objective='val_accuracy',
    max_trials=3,
    directory='Keras Tuner',
    project_name='activation_tuning'
)


In [ ]:
activation_tuner.search(
    xtrain,
    ytrain,
    validation_split=0.2,
    epochs=10,
    verbose=1
)


Trial 3 Complete [00h 00m 05s]
val_accuracy: 0.9780219793319702

Best val_accuracy So Far: 0.9780219793319702
Total elapsed time: 00h 00m 13s


In [ ]:

best_activation = activation_tuner.get_best_hyperparameters(1)[0].get('activation_function')

print("\nBest Activation Function:", best_activation)


Best Activation Function: sigmoid
